In [1]:
"""
Step 4b: Event-synchronized averaging analysis (Analysis 4).

Pools all 11 events together on a common "days from event" axis and asks:
averaged across very different incidents and years, is there a consistent
burnout response pattern around major cybersecurity events?

Automatically runs for every "*_resid_z" metric found in event_study_aligned.csv
-- so if your daily_burnout_series.csv has construct columns (EX/EMO/COG/MD),
this will produce a separate trajectory + test for each one, not just the
aggregate bat_score thresholds.

Two things this script does per metric:
  1. Plots the pooled average trajectory (mean + 95% CI band) for day -30 to
     +30, so you can see the shape of the response.
  2. Runs a paired test: for each event, compute (mean of post-event days) -
     (mean of pre-event days). With only 11 events, a Wilcoxon signed-rank test
     is used instead of a paired t-test, since it doesn't assume the 11 deltas
     are normally distributed.

Reads event_study_aligned.csv. No raw file access here.
"""

import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
import os

OUT_DIR = "/Users/nadia/Desktop/redditRun_june/event_study_v2/"
ALIGNED_CSV = os.path.join(OUT_DIR, "event_study_aligned.csv")
ALPHA = 0.05


def plot_pooled_trajectory(df, metric, out_path):
    grouped = df.groupby("days_from_event")[metric].agg(["mean", "sem", "count"])
    grouped["ci95"] = 1.96 * grouped["sem"]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(grouped.index, grouped["mean"], color="#2b6cb0", linewidth=2, label="pooled mean")
    ax.fill_between(
        grouped.index,
        grouped["mean"] - grouped["ci95"],
        grouped["mean"] + grouped["ci95"],
        color="#2b6cb0", alpha=0.2, label="95% CI"
    )
    ax.axvline(0, color="black", linestyle="--", linewidth=1, label="event day")
    ax.axhline(0, color="gray", linestyle=":", linewidth=1)
    ax.set_xlabel("Days from event")
    ax.set_ylabel(f"{metric}\n(z-score vs. own pre-event baseline)")
    ax.set_title(f"Event-synchronized average: {metric}\n(pooled across {df['event_name'].nunique()} events)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"  Saved plot -> {out_path}")


def paired_pre_post_test(df, metric):
    deltas = []
    per_event_rows = []
    for event_name, event_df in df.groupby("event_name"):
        pre = event_df.loc[event_df["days_from_event"] < 0, metric].mean()
        post = event_df.loc[event_df["days_from_event"] > 0, metric].mean()
        delta = post - pre
        deltas.append(delta)
        per_event_rows.append({"event_name": event_name, "pre_mean": pre, "post_mean": post, "delta": delta})

    stat, pvalue = wilcoxon(deltas)
    sig = "SIGNIFICANT" if pvalue < ALPHA else "not significant"
    print(f"\n  Paired pre/post Wilcoxon test for {metric}:")
    print(f"    median delta = {np.median(deltas):.3f}, W={stat:.3f}, p={pvalue:.4g} -> {sig} (alpha={ALPHA})")
    return pd.DataFrame(per_event_rows), pvalue


def main():
    if not os.path.exists(ALIGNED_CSV):
        print(f"Could not find {ALIGNED_CSV}. Run build_event_study_data.py first.")
        return

    df = pd.read_csv(ALIGNED_CSV)
    print(f"Loaded {len(df)} rows across {df['event_name'].nunique()} events")

    metrics = [c for c in df.columns if c.endswith("_resid_z")]
    if not metrics:
        print("No '*_resid_z' columns found -- check that build_event_study_data.py ran correctly.")
        return
    print(f"Metrics to analyze: {metrics}")

    summary_rows = []
    for metric in metrics:
        print(f"\n{'=' * 70}\n{metric}\n{'=' * 70}")

        plot_path = os.path.join(OUT_DIR, f"event_study_{metric}.png")
        plot_pooled_trajectory(df, metric, plot_path)

        per_event_df, pvalue = paired_pre_post_test(df, metric)
        per_event_path = os.path.join(OUT_DIR, f"event_study_per_event_{metric}.csv")
        per_event_df.to_csv(per_event_path, index=False)
        print(f"  Saved per-event deltas -> {per_event_path}")

        summary_rows.append({"metric": metric, "wilcoxon_p": pvalue})

    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(OUT_DIR, "event_study_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"\nSaved summary -> {summary_path}")

    print(f"\n{'=' * 70}\nHOW TO READ THIS\n{'=' * 70}")
    print("Each event's window is normalized to its OWN pre-event baseline before")
    print("pooling, so a 2018 event and a 2026 event are on the same scale even")
    print("though raw burnout volume grew a lot over the study period.")
    print("\nThe pooled trajectory plot shows the average shape of the response:")
    print("look for a rise around day 0 and how long it takes to decay back down")
    print("(the 'hangover window' from your Analysis 6 framing).")
    print("\nIf you have construct-level metrics (EX/EMO/COG/MD), compare their")
    print("plots side by side -- this is where you could show MD spiking harder or")
    print("longer than EX/EMO/COG around events, which would be a genuinely novel")
    print("finding tied to your compositional-shift argument.")
    print("\nThe Wilcoxon test asks a simpler yes/no question: across the 11 events,")
    print("is post-event activity reliably higher than pre-event activity? With")
    print("only 11 events this test has limited power -- a null result doesn't rule")
    print("out a real effect, it just means 11 events isn't a lot of data.")


if __name__ == "__main__":
    main()

Loaded 671 rows across 11 events
Metrics to analyze: ['n_burnout_resid_z', 'n_EX_resid_z', 'n_EMO_resid_z', 'n_COG_resid_z', 'n_MD_resid_z']

n_burnout_resid_z
  Saved plot -> /Users/nadia/Desktop/redditRun_june/event_study_v2/event_study_n_burnout_resid_z.png

  Paired pre/post Wilcoxon test for n_burnout_resid_z:
    median delta = 0.077, W=29.000, p=0.7646 -> not significant (alpha=0.05)
  Saved per-event deltas -> /Users/nadia/Desktop/redditRun_june/event_study_v2/event_study_per_event_n_burnout_resid_z.csv

n_EX_resid_z
  Saved plot -> /Users/nadia/Desktop/redditRun_june/event_study_v2/event_study_n_EX_resid_z.png

  Paired pre/post Wilcoxon test for n_EX_resid_z:
    median delta = -0.008, W=32.000, p=0.9658 -> not significant (alpha=0.05)
  Saved per-event deltas -> /Users/nadia/Desktop/redditRun_june/event_study_v2/event_study_per_event_n_EX_resid_z.csv

n_EMO_resid_z
  Saved plot -> /Users/nadia/Desktop/redditRun_june/event_study_v2/event_study_n_EMO_resid_z.png

  Paired pre/